In [0]:
# Databricks notebook source

from __future__ import annotations

import uuid

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")
catalog = dbutils.widgets.get("catalog").strip()

# ============================================================
# Constants
# ============================================================
SCRIPT_NAME = "300_build_gold_dim_date.py"
RUN_ID = str(uuid.uuid4())

CONFIG_HOLIDAY_TABLE = f"{catalog}.gold.config_holiday"
DIM_DATE_TABLE = f"{catalog}.gold.dim_date"
RUN_LOG_TABLE = f"{catalog}.staging.pipeline_run_log"

# ============================================================
# Helpers
# ============================================================
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def log_run(status: str, row_count: int | None, message: str) -> None:
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE}
        VALUES (
              '{RUN_ID}'
            , '{SCRIPT_NAME}'
            , '{sql_escape(status)}'
            , {row_count_sql}
            , '{sql_escape(message)}'
            , current_timestamp()
        )
    """)

# ============================================================
# Build gold.config_holiday
# ============================================================
try:
    spark.sql(f"DROP VIEW IF EXISTS {DIM_DATE_TABLE}")
    spark.sql(f"DROP TABLE IF EXISTS {DIM_DATE_TABLE}")
    spark.sql(f"DROP TABLE IF EXISTS {CONFIG_HOLIDAY_TABLE}")

    spark.sql(f"""
    CREATE TABLE {CONFIG_HOLIDAY_TABLE} (
          holiday_key INT
        , holiday_name STRING
        , holiday_type INT
        , year_start INT
        , year_end INT
        , month INT
        , day INT
        , day_of_week INT
        , week_of_month INT
        , description STRING
        , country_code STRING
        , multiple_observances BOOLEAN
        , observance_dates STRING
        , holiday_group STRING
        , valid_through DATE
    )
    """)

    spark.sql(f"""
    INSERT INTO {CONFIG_HOLIDAY_TABLE} (
          holiday_key
        , holiday_name
        , holiday_type
        , year_start
        , year_end
        , month
        , day
        , day_of_week
        , week_of_month
        , description
        , country_code
        , multiple_observances
        , observance_dates
        , holiday_group
        , valid_through
    )
    VALUES
          (1,  'New Year''s Day',             1, 1900, NULL,  1,  1, NULL, NULL, 'New Year holiday',              'US', FALSE, NULL, 'Federal', NULL)
        , (2,  'Martin Luther King Jr. Day',  2, 1986, NULL,  1, NULL,    1,    3, 'Third Monday in January',     'US', FALSE, NULL, 'Federal', NULL)
        , (3,  'Presidents Day',              2, 1971, NULL,  2, NULL,    1,    3, 'Third Monday in February',    'US', FALSE, NULL, 'Federal', NULL)
        , (4,  'Memorial Day',                2, 1971, NULL,  5, NULL,    1,   -1, 'Last Monday in May',          'US', FALSE, NULL, 'Federal', NULL)
        , (5,  'Independence Day',            1, 1776, NULL,  7,  4, NULL, NULL, 'Independence Day',              'US', FALSE, NULL, 'Federal', NULL)
        , (6,  'Labor Day',                   2, 1894, NULL,  9, NULL,    1,    1, 'First Monday in September',   'US', FALSE, NULL, 'Federal', NULL)
        , (7,  'Columbus Day',                2, 1971, NULL, 10, NULL,    1,    2, 'Second Monday in October',    'US', FALSE, NULL, 'Federal', NULL)
        , (8,  'Veterans Day',                1, 1919, NULL, 11, 11, NULL, NULL, 'Veterans Day',                 'US', FALSE, NULL, 'Federal', NULL)
        , (9,  'Thanksgiving Day',            2, 1941, NULL, 11, NULL,    4,    4, 'Fourth Thursday in November', 'US', FALSE, NULL, 'Federal', NULL)
        , (10, 'Christmas Day',               1, 1870, NULL, 12, 25, NULL, NULL, 'Christmas Day',                'US', FALSE, NULL, 'Federal', NULL)
    """)

    # ========================================================
    # Build gold.dim_date
    # ========================================================
    spark.sql(f"""
    CREATE TABLE {DIM_DATE_TABLE} AS
    WITH date_sequence AS (
        SELECT explode(sequence(to_date('2010-01-01'), to_date('2050-12-31'), interval 1 day)) AS calendar_date
    )

    , base_dates AS (
        SELECT
              calendar_date
            , CAST(date_format(calendar_date, 'yyyyMMdd') AS INT) AS date_key
            , date_format(calendar_date, 'EEEE') AS day_name
            , date_format(calendar_date, 'E') AS day_name_short
            , day(calendar_date) AS day_of_month
            , quarter(calendar_date) AS quarter
            , month(calendar_date) AS month
            , year(calendar_date) AS year
            , dayofyear(calendar_date) AS day_of_year
            , weekofyear(calendar_date) AS week_of_year
            , trunc(calendar_date, 'MM') AS first_day_of_month
            , last_day(calendar_date) AS last_day_of_month
            , trunc(calendar_date, 'QUARTER') AS first_day_of_quarter
            , date_sub(add_months(trunc(calendar_date, 'QUARTER'), 3), 1) AS last_day_of_quarter
            , trunc(calendar_date, 'YEAR') AS first_day_of_year
            , date_sub(add_months(trunc(calendar_date, 'YEAR'), 12), 1) AS last_day_of_year
            , date_format(calendar_date, 'MMMM') AS month_name
            , date_format(calendar_date, 'MMM') AS month_name_short
            , upper(concat(substr(date_format(calendar_date, 'MMM'), 1, 1), lower(substr(date_format(calendar_date, 'MMM'), 2)), '-', cast(year(calendar_date) as string))) AS month_year
            , concat('Q', cast(quarter(calendar_date) as string), '-', cast(year(calendar_date) as string)) AS period
            , concat(cast(year(calendar_date) as string), '-Q', cast(quarter(calendar_date) as string)) AS year_quarter
            , concat('Q', cast(quarter(calendar_date) as string)) AS quarter_label
            , CAST(date_format(calendar_date, 'yyyyMM') AS INT) AS year_month_key
            , (year(calendar_date) * 100) + month(calendar_date) AS month_sort
            , CASE WHEN dayofweek(calendar_date) IN (1, 7) THEN 0 ELSE 1 END AS flag_weekday
            , CASE WHEN calendar_date = last_day(calendar_date) THEN 1 ELSE 0 END AS flag_last_day_of_month
            , pmod(dayofweek(calendar_date) + 5, 7) + 1 AS day_of_week_iso
        FROM date_sequence
    )

    , fiscal_dates AS (
        SELECT
              *
            , CASE WHEN month >= 4 THEN year + 1 ELSE year END AS fiscal_year
            , CASE
                  WHEN month BETWEEN 4 AND 6 THEN 1
                  WHEN month BETWEEN 7 AND 9 THEN 2
                  WHEN month BETWEEN 10 AND 12 THEN 3
                  ELSE 4
              END AS fiscal_quarter
        FROM base_dates
    )

    , fixed_holidays AS (
        SELECT
              d.calendar_date
            , h.holiday_key
            , h.holiday_name
            , h.description AS holiday_description
            , h.country_code
            , h.holiday_group
        FROM fiscal_dates d
        LEFT JOIN {CONFIG_HOLIDAY_TABLE} h
            ON h.holiday_type = 1
           AND d.month = h.month
           AND d.day_of_month = h.day
           AND d.year BETWEEN h.year_start AND COALESCE(h.year_end, 9999)
    )

    SELECT
          d.calendar_date
        , d.date_key
        , d.day_name
        , d.day_name_short
        , d.day_of_month
        , d.day_of_week_iso
        , d.quarter
        , d.quarter_label
        , d.month
        , d.year
        , d.day_of_year
        , d.week_of_year
        , d.first_day_of_month
        , d.last_day_of_month
        , d.first_day_of_quarter
        , d.last_day_of_quarter
        , d.first_day_of_year
        , d.last_day_of_year
        , d.month_name
        , d.month_name_short
        , d.month_year
        , d.period
        , d.year_quarter
        , d.year_month_key
        , d.month_sort
        , d.flag_weekday
        , d.flag_last_day_of_month
        , d.fiscal_year
        , d.fiscal_quarter
        , concat('FY', cast(d.fiscal_year as string)) AS fiscal_year_label
        , concat('FY', cast(d.fiscal_year as string), '-Q', cast(d.fiscal_quarter as string)) AS fiscal_quarter_label
        , CASE WHEN h.holiday_name IS NOT NULL THEN 1 ELSE 0 END AS is_holiday
        , h.holiday_key
        , h.holiday_name
        , h.holiday_description
        , h.country_code AS holiday_country_code
        , h.holiday_group
    FROM fiscal_dates d
    LEFT JOIN fixed_holidays h
        ON d.calendar_date = h.calendar_date
    """)

    row_count = spark.sql(f"""
        SELECT COUNT(*) AS row_count
        FROM {DIM_DATE_TABLE}
    """).collect()[0]["row_count"]

    log_run(
        "SUCCESS",
        row_count,
        "Built gold.config_holiday and gold.dim_date successfully."
    )

    print("=" * 70)
    print("Built gold.config_holiday")
    print("Built gold.dim_date")
    print(f"Row count: {row_count:,}")
    print("=" * 70)

except Exception as exc:
    log_run("FAILED", None, str(exc))
    raise